In [1]:
# patch tacred data to retacred

In [7]:
%run ./utils/patch_tacred_2_retacred.py

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15509/15509 [00:01<00:00, 13435.60it/s]


In [8]:
%mkdir ../models
%cd ../models

/home/jon/uom-relation-extraction/notebooks/models


In [10]:
!git clone git@github.com:qipeng/gcn-over-pruned-trees.git

Cloning into 'gcn-over-pruned-trees'...
remote: Enumerating objects: 56, done.
remote: Total 56 (delta 0), reused 0 (delta 0), pack-reused 56 (from 1)
Receiving objects: 100% (56/56), 600.56 KiB | 1.80 MiB/s, done.
Resolving deltas: 100% (16/16), done.


In [11]:
%cd gcn-over-pruned-trees/

/home/jon/uom-relation-extraction/notebooks/models/gcn-over-pruned-trees


In [12]:
# download and unzip GloVe vectors from the Stanford NLP group website

In [13]:
!chmod +x download.sh; ./download.sh

==> Downloading glove vectors...
--2025-02-18 19:21:03--  http://nlp.stanford.edu/data/glove.840B.300d.zip
171.64.67.140.stanford.edu (nlp.stanford.edu)... 
connected. to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... 
302 Foundest sent, awaiting response... 
Location: https://nlp.stanford.edu/data/glove.840B.300d.zip [following]
--2025-02-18 19:21:03--  https://nlp.stanford.edu/data/glove.840B.300d.zip
connected. to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... 
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.840B.300d.zip [following]
--2025-02-18 19:21:04--  https://downloads.cs.stanford.edu/nlp/data/glove.840B.300d.zip
171.64.64.22wnloads.cs.stanford.edu (downloads.cs.stanford.edu)... 
connected. to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... 
HTTP request sent, awaiting response... 200 OK
Length: 2176768927 (2.0G) [application/zip]
Saving to: ‘glove.840B.300

In [14]:
# remove sample data

In [15]:
!pwd

/home/jon/uom-relation-extraction/notebooks/models/gcn-over-pruned-trees


In [16]:
!rm dataset/tacred/*.json

In [17]:
# move original tacred dataset into data dir

In [18]:
!pwd

/home/jon/uom-relation-extraction/notebooks/models/gcn-over-pruned-trees


In [23]:
!cp ../../../data/tacred/*.json dataset/tacred/.

In [24]:
!ls dataset/tacred/.

dev.json  README.md  test.json	train.json


In [25]:
%run prepare_vocab.py dataset/tacred dataset/vocab --glove_dir dataset/glove

Directory dataset/vocab do not exist; creating...
loading files...
2304974 tokens from 68124 examples loaded from dataset/tacred/train.json.
727556 tokens from 22631 examples loaded from dataset/tacred/dev.json.
490062 tokens from 15509 examples loaded from dataset/tacred/test.json.
loading glove...
2195892 words loaded from glove.
building vocab...
vocab built with 53953/59420 words.
calculating oov...
train oov: 17631/2304974 (0.76%)
dev oov: 33664/727556 (4.63%)
test oov: 24310/490062 (4.96%)
building embeddings...
embedding size: 53953 x 300
dumping to files...
all done.


In [27]:
!nvidia-smi

Tue Feb 18 19:35:41 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.86.15              Driver Version: 570.86.15      CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 4000 SFF Ada ...    Off |   00000000:01:00.0  On |                  Off |
| 30%   39C    P5             15W /   70W |    1452MiB /  20475MiB |     36%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [29]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.current_device())

True
0


In [30]:
torch.cuda.get_device_name(0)

'NVIDIA RTX 4000 SFF Ada Generation'

In [32]:
# train_gcn.sh and train_cgcn.sh contain the following config:
# parser.add_argument('--cuda', type=bool, default=torch.cuda.is_available())
# parser.add_argument('--cpu', action='store_true', help='Ignore CUDA.')

In [33]:
# training gcn on original TACRED dataset:

In [34]:
!bash train_gcn.sh 0

Vocab size 53953 loaded from file
Loading data from dataset/tacred with batch size 50...
1363 batches created for dataset/tacred/train.json
453 batches created for dataset/tacred/dev.json
Directory ./saved_models/00 do not exist; creating...
Config saved to file ./saved_models/00/config.json

Running with the following configs:
	data_dir : dataset/tacred
	vocab_dir : dataset/vocab
	emb_dim : 300
	ner_dim : 30
	pos_dim : 30
	hidden_dim : 200
	num_layers : 2
	input_dropout : 0.5
	gcn_dropout : 0.5
	word_dropout : 0.04
	topn : 10000000000.0
	lower : False
	prune_k : 1
	conv_l2 : 0
	pooling : max
	pooling_l2 : 0.003
	mlp_layers : 2
	no_adj : False
	rnn : False
	rnn_hidden : 200
	rnn_layers : 1
	rnn_dropout : 0.5
	lr : 0.3
	lr_decay : 0.9
	decay_epoch : 5
	optim : sgd
	num_epoch : 100
	batch_size : 50
	max_grad_norm : 5.0
	log_step : 20
	log : logs.txt
	save_epoch : 100
	save_dir : ./saved_models
	id : 0
	info : 
	seed : 0
	cuda : True
	cpu : False
	load : False
	model_file : None
	num_clas

In [35]:
# evaluating gcn 0
%run eval.py saved_models/00 --dataset test

Loading model from saved_models/00/best_model.pt


/home/jon/uom-relation-extraction/notebooks/models/gcn-over-pruned-trees/utils/torch_utils.py:158: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dump = torch.load(filename)


Finetune all embeddings.


/home/jon/uom-relation-extraction/notebooks/models/gcn-over-pruned-trees/model/trainer.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filename)

Vocab size 53953 loaded from file
Loading data from dataset/tacred/test.json with batch size 50...
311 batches created for dataset/tacred/test.json

Running with the following configs:
	data_dir : dataset/tacred
	vocab_dir : dataset/vocab
	emb_dim : 300
	ner_dim : 30
	pos_dim : 30
	hidden_dim : 200
	num_layers : 2
	input_dropout : 0.5
	gcn_dropout : 0.5
	word_dropout : 0.04
	topn : 10000000000.0
	lower : False
	prune_k : 1
	conv_l2 : 0
	pooling : max
	pooling_l2 : 0.003
	mlp_layers : 2
	no_adj : False
	rnn : False
	rnn_hidden : 200
	rnn_layers : 1
	rnn_dropout : 0.5
	lr : 0.3
	lr_decay : 0.9
	decay_epoch : 5
	optim : sgd
	num_epoch : 100
	batch_size : 50
	max_grad_norm : 5.0
	log_step : 20
	log : logs.txt
	save_epoch : 100
	save_dir : ./saved_models
	id : 0
	info : 
	seed : 0
	cuda : True
	cpu : False
	load : False
	model_file : None
	num_class : 42
	vocab_size : 53953
	model_save_dir : ./saved_models/00




100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 311/311 [00:02<00:00, 122.34it/s]

Per-relation statistics:
org:alternate_names                  P:  71.04%  R:  73.71%  F1:  72.35%  #: 213
org:city_of_headquarters             P:  76.39%  R:  67.07%  F1:  71.43%  #: 82
org:country_of_headquarters          P:  64.58%  R:  28.70%  F1:  39.74%  #: 108
org:dissolved                        P: 100.00%  R:   0.00%  F1:   0.00%  #: 2
org:founded                          P:  83.87%  R:  70.27%  F1:  76.47%  #: 37
org:founded_by                       P:  80.77%  R:  30.88%  F1:  44.68%  #: 68
org:member_of                        P: 100.00%  R:   0.00%  F1:   0.00%  #: 18
org:members                          P: 100.00%  R:   0.00%  F1:   0.00%  #: 31
org:number_of_employees/members      P:  80.00%  R:  42.11%  F1:  55.17%  #: 19
org:parents                          P:  28.57%  R:   3.23%  F1:   5.80%  #: 62
org:political/religious_affiliation  P:  62.50%  R:  50.00%  F1:  55.56%  #: 10
org:shareholders                     P: 100.00%  R:  23.08%  F1:  37.50%  #: 13
org:stateorpro

### gcn results on TACRED
    
    Per-relation statistics:
    org:alternate_names                  P:  71.04%  R:  73.71%  F1:  72.35%  #: 213
    org:city_of_headquarters             P:  76.39%  R:  67.07%  F1:  71.43%  #: 82
    org:country_of_headquarters          P:  64.58%  R:  28.70%  F1:  39.74%  #: 108
    org:dissolved                        P: 100.00%  R:   0.00%  F1:   0.00%  #: 2
    org:founded                          P:  83.87%  R:  70.27%  F1:  76.47%  #: 37
    org:founded_by                       P:  80.77%  R:  30.88%  F1:  44.68%  #: 68
    org:member_of                        P: 100.00%  R:   0.00%  F1:   0.00%  #: 18
    org:members                          P: 100.00%  R:   0.00%  F1:   0.00%  #: 31
    org:number_of_employees/members      P:  80.00%  R:  42.11%  F1:  55.17%  #: 19
    org:parents                          P:  28.57%  R:   3.23%  F1:   5.80%  #: 62
    org:political/religious_affiliation  P:  62.50%  R:  50.00%  F1:  55.56%  #: 10
    org:shareholders                     P: 100.00%  R:  23.08%  F1:  37.50%  #: 13
    org:stateorprovince_of_headquarters  P:  71.15%  R:  72.55%  F1:  71.84%  #: 51
    org:subsidiaries                     P:  55.56%  R:  22.73%  F1:  32.26%  #: 44
    org:top_members/employees            P:  67.87%  R:  81.79%  F1:  74.18%  #: 346
    org:website                          P:  50.00%  R:  96.15%  F1:  65.79%  #: 26
    per:age                              P:  77.05%  R:  94.00%  F1:  84.68%  #: 200
    per:alternate_names                  P:   0.00%  R:   0.00%  F1:   0.00%  #: 11
    per:cause_of_death                   P:  73.33%  R:  21.15%  F1:  32.84%  #: 52
    per:charges                          P:  66.13%  R:  79.61%  F1:  72.25%  #: 103
    per:children                         P:  29.17%  R:  18.92%  F1:  22.95%  #: 37
    per:cities_of_residence              P:  59.06%  R:  46.56%  F1:  52.07%  #: 189
    per:city_of_birth                    P:  28.57%  R:  40.00%  F1:  33.33%  #: 5
    per:city_of_death                    P: 100.00%  R:  21.43%  F1:  35.29%  #: 28
    per:countries_of_residence           P:  50.00%  R:  35.14%  F1:  41.27%  #: 148
    per:country_of_birth                 P: 100.00%  R:   0.00%  F1:   0.00%  #: 5
    per:country_of_death                 P: 100.00%  R:   0.00%  F1:   0.00%  #: 9
    per:date_of_birth                    P:  75.00%  R:  66.67%  F1:  70.59%  #: 9
    per:date_of_death                    P:  75.00%  R:  27.78%  F1:  40.54%  #: 54
    per:employee_of                      P:  68.02%  R:  63.64%  F1:  65.75%  #: 264
    per:origin                           P:  69.42%  R:  63.64%  F1:  66.40%  #: 132
    per:other_family                     P: 100.00%  R:   0.00%  F1:   0.00%  #: 60
    per:parents                          P:  58.49%  R:  35.23%  F1:  43.97%  #: 88
    per:religion                         P:  61.70%  R:  61.70%  F1:  61.70%  #: 47
    per:schools_attended                 P:  77.27%  R:  56.67%  F1:  65.38%  #: 30
    per:siblings                         P:  78.57%  R:  60.00%  F1:  68.04%  #: 55
    per:spouse                           P:  51.85%  R:  63.64%  F1:  57.14%  #: 66
    per:stateorprovince_of_birth         P:  40.00%  R:  50.00%  F1:  44.44%  #: 8
    per:stateorprovince_of_death         P:  75.00%  R:  21.43%  F1:  33.33%  #: 14
    per:stateorprovinces_of_residence    P:  58.57%  R:  50.62%  F1:  54.30%  #: 81
    per:title                            P:  81.19%  R:  79.40%  F1:  80.28%  #: 500
    
    Final Score:
    Precision (micro): 69.063%
       Recall (micro): 59.218%
           F1 (micro): 63.763%
    test set evaluate result: 0.69	0.59	0.64


### Training CGCN on TACRED

In [37]:
!bash train_cgcn.sh 1

Vocab size 53953 loaded from file
Loading data from dataset/tacred with batch size 50...
1363 batches created for dataset/tacred/train.json
453 batches created for dataset/tacred/dev.json
Directory ./saved_models/01 do not exist; creating...
Config saved to file ./saved_models/01/config.json

Running with the following configs:
	data_dir : dataset/tacred
	vocab_dir : dataset/vocab
	emb_dim : 300
	ner_dim : 30
	pos_dim : 30
	hidden_dim : 200
	num_layers : 2
	input_dropout : 0.5
	gcn_dropout : 0.5
	word_dropout : 0.04
	topn : 10000000000.0
	lower : False
	prune_k : 1
	conv_l2 : 0
	pooling : max
	pooling_l2 : 0.003
	mlp_layers : 2
	no_adj : False
	rnn : True
	rnn_hidden : 200
	rnn_layers : 1
	rnn_dropout : 0.5
	lr : 0.3
	lr_decay : 0.9
	decay_epoch : 5
	optim : sgd
	num_epoch : 100
	batch_size : 50
	max_grad_norm : 5.0
	log_step : 20
	log : logs.txt
	save_epoch : 100
	save_dir : ./saved_models
	id : 1
	info : 
	seed : 0
	cuda : True
	cpu : False
	load : False
	model_file : None
	num_class

In [38]:
# evaluating gcn 1
%run eval.py saved_models/01 --dataset test

Loading model from saved_models/01/best_model.pt
Finetune all embeddings.


/home/jon/uom-relation-extraction/notebooks/models/gcn-over-pruned-trees/utils/torch_utils.py:158: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dump = torch.load(filename)


Vocab size 53953 loaded from file
Loading data from dataset/tacred/test.json with batch size 50...
311 batches created for dataset/tacred/test.json

Running with the following configs:
	data_dir : dataset/tacred
	vocab_dir : dataset/vocab
	emb_dim : 300
	ner_dim : 30
	pos_dim : 30
	hidden_dim : 200
	num_layers : 2
	input_dropout : 0.5
	gcn_dropout : 0.5
	word_dropout : 0.04
	topn : 10000000000.0
	lower : False
	prune_k : 1
	conv_l2 : 0
	pooling : max
	pooling_l2 : 0.003
	mlp_layers : 2
	no_adj : False
	rnn : True
	rnn_hidden : 200
	rnn_layers : 1
	rnn_dropout : 0.5
	lr : 0.3
	lr_decay : 0.9
	decay_epoch : 5
	optim : sgd
	num_epoch : 100
	batch_size : 50
	max_grad_norm : 5.0
	log_step : 20
	log : logs.txt
	save_epoch : 100
	save_dir : ./saved_models
	id : 1
	info : 
	seed : 0
	cuda : True
	cpu : False
	load : False
	model_file : None
	num_class : 42
	vocab_size : 53953
	model_save_dir : ./saved_models/01




100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 311/311 [00:03<00:00, 99.48it/s]

Per-relation statistics:
org:alternate_names                  P:  76.82%  R:  84.04%  F1:  80.27%  #: 213
org:city_of_headquarters             P:  77.27%  R:  62.20%  F1:  68.92%  #: 82
org:country_of_headquarters          P:  66.67%  R:  27.78%  F1:  39.22%  #: 108
org:dissolved                        P: 100.00%  R:   0.00%  F1:   0.00%  #: 2
org:founded                          P:  88.57%  R:  83.78%  F1:  86.11%  #: 37
org:founded_by                       P:  70.00%  R:  61.76%  F1:  65.62%  #: 68
org:member_of                        P: 100.00%  R:   0.00%  F1:   0.00%  #: 18
org:members                          P:   0.00%  R:   0.00%  F1:   0.00%  #: 31
org:number_of_employees/members      P:  63.64%  R:  73.68%  F1:  68.29%  #: 19
org:parents                          P:  46.15%  R:  19.35%  F1:  27.27%  #: 62
org:political/religious_affiliation  P:  29.63%  R:  80.00%  F1:  43.24%  #: 10
org:shareholders                     P:  75.00%  R:  23.08%  F1:  35.29%  #: 13
org:stateorpro

### CGCN Results on TACRED
    
    Per-relation statistics:
    org:alternate_names                  P:  76.82%  R:  84.04%  F1:  80.27%  #: 213
    org:city_of_headquarters             P:  77.27%  R:  62.20%  F1:  68.92%  #: 82
    org:country_of_headquarters          P:  66.67%  R:  27.78%  F1:  39.22%  #: 108
    org:dissolved                        P: 100.00%  R:   0.00%  F1:   0.00%  #: 2
    org:founded                          P:  88.57%  R:  83.78%  F1:  86.11%  #: 37
    org:founded_by                       P:  70.00%  R:  61.76%  F1:  65.62%  #: 68
    org:member_of                        P: 100.00%  R:   0.00%  F1:   0.00%  #: 18
    org:members                          P:   0.00%  R:   0.00%  F1:   0.00%  #: 31
    org:number_of_employees/members      P:  63.64%  R:  73.68%  F1:  68.29%  #: 19
    org:parents                          P:  46.15%  R:  19.35%  F1:  27.27%  #: 62
    org:political/religious_affiliation  P:  29.63%  R:  80.00%  F1:  43.24%  #: 10
    org:shareholders                     P:  75.00%  R:  23.08%  F1:  35.29%  #: 13
    org:stateorprovince_of_headquarters  P:  72.34%  R:  66.67%  F1:  69.39%  #: 51
    org:subsidiaries                     P:  51.85%  R:  31.82%  F1:  39.44%  #: 44
    org:top_members/employees            P:  71.82%  R:  83.24%  F1:  77.11%  #: 346
    org:website                          P:  55.81%  R:  92.31%  F1:  69.57%  #: 26
    per:age                              P:  85.07%  R:  94.00%  F1:  89.31%  #: 200
    per:alternate_names                  P:   0.00%  R:   0.00%  F1:   0.00%  #: 11
    per:cause_of_death                   P:  70.00%  R:  26.92%  F1:  38.89%  #: 52
    per:charges                          P:  69.05%  R:  84.47%  F1:  75.98%  #: 103
    per:children                         P:  39.39%  R:  35.14%  F1:  37.14%  #: 37
    per:cities_of_residence              P:  59.65%  R:  53.97%  F1:  56.67%  #: 189
    per:city_of_birth                    P:  25.00%  R:  20.00%  F1:  22.22%  #: 5
    per:city_of_death                    P:  90.00%  R:  32.14%  F1:  47.37%  #: 28
    per:countries_of_residence           P:  55.74%  R:  45.95%  F1:  50.37%  #: 148
    per:country_of_birth                 P: 100.00%  R:   0.00%  F1:   0.00%  #: 5
    per:country_of_death                 P: 100.00%  R:   0.00%  F1:   0.00%  #: 9
    per:date_of_birth                    P: 100.00%  R:  66.67%  F1:  80.00%  #: 9
    per:date_of_death                    P:  66.67%  R:  25.93%  F1:  37.33%  #: 54
    per:employee_of                      P:  68.70%  R:  68.18%  F1:  68.44%  #: 264
    per:origin                           P:  69.03%  R:  59.09%  F1:  63.67%  #: 132
    per:other_family                     P:  60.00%  R:  15.00%  F1:  24.00%  #: 60
    per:parents                          P:  72.88%  R:  48.86%  F1:  58.50%  #: 88
    per:religion                         P:  56.86%  R:  61.70%  F1:  59.18%  #: 47
    per:schools_attended                 P:  68.00%  R:  56.67%  F1:  61.82%  #: 30
    per:siblings                         P:  70.83%  R:  61.82%  F1:  66.02%  #: 55
    per:spouse                           P:  52.17%  R:  72.73%  F1:  60.76%  #: 66
    per:stateorprovince_of_birth         P:  44.44%  R:  50.00%  F1:  47.06%  #: 8
    per:stateorprovince_of_death         P:  75.00%  R:  21.43%  F1:  33.33%  #: 14
    per:stateorprovinces_of_residence    P:  64.18%  R:  53.09%  F1:  58.11%  #: 81
    per:title                            P:  81.45%  R:  80.80%  F1:  81.12%  #: 500
    
    Final Score:
    Precision (micro): 70.378%
       Recall (micro): 63.880%
           F1 (micro): 66.971%
    test set evaluate result: 0.70	0.64	0.67
    Evaluation ended.

### Applying Patches

**see utils directory for cgcn_patch_*.py files**

* cgcn_patch_train.py -> cgcn_dir/train.py
* cgcn_patch_scorer.py -> cgcn_dir/utils/scorer.py
* cgcn_patch_category_maps.py -> cgcn_dir/utils/category_maps.py
* cgcn_patch_constant.py -> cgcn_dir/utils/constant.py

Then we drop the ReTACRED data a retacred folder within cgcn_dir/data/retacred, update the train.py parameters & reconstruct/retune the embeddings.

*note: parts of the above instructions were completed within a terminal window, opps!*

In [39]:
!pwd

/home/jon/uom-relation-extraction/notebooks/models/gcn-over-pruned-trees


In [41]:
!ls dataset/

glove  retacred  tacred  vocab_TACRED


In [42]:
!ls dataset/retacred

dev_full.json  dev.json  test_full.json  test.json  train_full.json  train.json


In [44]:
%run prepare_vocab.py dataset/retacred dataset/vocab --glove_dir dataset/glove

Directory dataset/vocab do not exist; creating...
loading files...
1973872 tokens from 58465 examples loaded from dataset/retacred/train_full.json.
627066 tokens from 19584 examples loaded from dataset/retacred/dev_full.json.
422256 tokens from 13418 examples loaded from dataset/retacred/test_full.json.
loading glove...
2195892 words loaded from glove.
building vocab...
vocab built with 50115/54597 words.
calculating oov...
train oov: 13242/1973872 (0.67%)
dev oov: 28651/627066 (4.57%)
test oov: 20568/422256 (4.87%)
building embeddings...
embedding size: 50115 x 300
dumping to files...
all done.


In [45]:
# hopefully now it just works..?

In [47]:
!bash train_gcn.sh 2

Vocab size 50115 loaded from file
Loading data from dataset/retacred with batch size 50...
1170 batches created for dataset/retacred/train_full.json
392 batches created for dataset/retacred/dev_full.json
269 batches created for dataset/retacred/test_full.json
Directory ./saved_models/02 do not exist; creating...
Config saved to file ./saved_models/02/config.json

Running with the following configs:
	data_dir : dataset/retacred
	vocab_dir : dataset/vocab
	emb_dim : 300
	ner_dim : 30
	pos_dim : 30
	hidden_dim : 200
	num_layers : 2
	input_dropout : 0.5
	gcn_dropout : 0.5
	word_dropout : 0.04
	topn : 10000000000.0
	lower : False
	prune_k : 1
	conv_l2 : 0
	pooling : max
	pooling_l2 : 0.003
	mlp_layers : 2
	no_adj : False
	rnn : False
	rnn_hidden : 200
	rnn_layers : 1
	rnn_dropout : 0.5
	lr : 0.3
	lr_decay : 0.9
	decay_epoch : 5
	optim : sgd
	num_epoch : 100
	batch_size : 50
	max_grad_norm : 5.0
	log_step : 20
	log : logs.txt
	save_epoch : 100
	save_dir : ./saved_models
	id : 2
	info : 
	see

In [51]:
# evaluating gcn 2 on RETACRED
%run eval.py saved_models/02 --dataset test_full

Loading model from saved_models/02/best_model.pt
Finetune all embeddings.
Vocab size 50115 loaded from file
Loading data from dataset/retacred/test_full.json with batch size 50...


KeyError: 'per:identity'

In [52]:
# I've broken something, ah I'll fix it tomorrow.